# google_cloud_runtimeconfig smoke test

Smoke test for `google-cloud-runtimeconfig`, the client for the Google Cloud Runtime Configurator.
Every example runs offline: it builds configs and variables with anonymous credentials, parses
variables from their API representation, and checks the client-side validation. No request is sent.

## Example 1: Client and Config construction

In [ ]:
# Example 1: Client and Config construction. AnonymousCredentials keep this offline: config() and variable()
# only build local objects; reload(), exists(), create() and update() are the calls that reach the service.
from google.auth.credentials import AnonymousCredentials
from google.cloud import runtimeconfig

client = runtimeconfig.Client(project="demo-project", credentials=AnonymousCredentials())
print(f"client: {type(client).__name__}, project: {client.project}")
assert client.project == "demo-project", client.project

config = client.config("demo")
print(f"config: {config!r}")
print(f"config full_name: {config.full_name}")
print(f"config path: {config.path}")
assert (config.name, config.project, config.client) == ("demo", "demo-project", client)
assert config.full_name == "projects/demo-project/configs/demo", config.full_name
assert config.path == "/projects/demo-project/configs/demo", config.path
assert config.description is None, "description is only filled in by reload()"

## Example 2: Variable names and paths

In [ ]:
# Example 2: Variables. Names may contain slashes, which the service treats as a hierarchy.
from google.auth.credentials import AnonymousCredentials
from google.cloud import runtimeconfig
from google.cloud.runtimeconfig.variable import Variable

client = runtimeconfig.Client(project="demo-project", credentials=AnonymousCredentials())
config = client.config("demo")

for name in ("timeout", "database/host", "feature/flags/new_ui"):
    var = config.variable(name)
    print(f"{name!r}: full_name={var.full_name}")
    assert isinstance(var, Variable) and var.config is config and var.client is client
    assert var.full_name == f"projects/demo-project/configs/demo/variables/{name}", var.full_name
    assert var.path == "/" + var.full_name, var.path

var = config.variable("a/b")
assert var.full_name.endswith("variables/a/b"), var.full_name
assert (var.text, var.value, var.update_time) == (None, None, None), "nothing is set until reload()"
assert var.state == "VARIABLE_STATE_UNSPECIFIED", var.state

## Example 3: Text variable from its API representation

In [ ]:
# Example 3: Parse a variable from its API representation (the JSON the service returns),
# here a text variable.
import datetime

from google.auth.credentials import AnonymousCredentials
from google.cloud import runtimeconfig
from google.cloud.runtimeconfig.variable import Variable

client = runtimeconfig.Client(project="demo-project", credentials=AnonymousCredentials())
config = client.config("demo")

resource = {
    "name": "projects/demo-project/configs/demo/variables/database/host",
    "text": "db.example.com",
    "state": "UPDATED",
    "updateTime": "2024-05-01T12:30:45.123456Z",
}
var = Variable.from_api_repr(resource, config)
print(f"name: {var.name}")
print(f"text: {var.text}")
print(f"state: {var.state}")
print(f"update_time: {var.update_time.isoformat()}")
assert var.name == "database/host", var.name
assert var.full_name == resource["name"], var.full_name
assert (var.text, var.value, var.state) == ("db.example.com", None, "UPDATED")
assert var.update_time == datetime.datetime(2024, 5, 1, 12, 30, 45, 123456, tzinfo=datetime.timezone.utc), var.update_time

## Example 4: Binary variable values

In [ ]:
# Example 4: Binary variables. The service sends `value` base64-encoded; the property decodes it to bytes.
import base64

from google.auth.credentials import AnonymousCredentials
from google.cloud import runtimeconfig
from google.cloud.runtimeconfig.variable import Variable

client = runtimeconfig.Client(project="demo-project", credentials=AnonymousCredentials())
config = client.config("demo")

payload = b"\x00\x01binary-config\xff"
resource = {
    "name": "projects/demo-project/configs/demo/variables/blob",
    "value": base64.b64encode(payload).decode("ascii"),
    "state": "UPDATED",
}
var = Variable.from_api_repr(resource, config)
print(f"name: {var.name}, value: {var.value!r}")
assert var.value == payload, var.value
assert var.text is None, var.text

## Example 5: text and value are mutually exclusive

In [ ]:
# Example 5: text and value are mutually exclusive on one variable; the client rejects setting both
# before any request is sent.
from google.auth.credentials import AnonymousCredentials
from google.cloud import runtimeconfig
from google.cloud.runtimeconfig.exceptions import Error

client = runtimeconfig.Client(project="demo-project", credentials=AnonymousCredentials())
config = client.config("demo")

text_var = config.variable("greeting")
text_var.text = "hello"
print(f"text variable: {text_var.name} = {text_var.text!r}")
assert text_var.text == "hello"
try:
    text_var.value = b"also bytes"
except Error as e:
    print(f"setting value on a text variable raised Error: {e}")
else:
    raise AssertionError("expected runtimeconfig Error for text + value")

value_var = config.variable("blob")
value_var.value = b"raw"
try:
    value_var.text = "also text"
except Error as e:
    print(f"setting text on a value variable raised Error: {e}")
else:
    raise AssertionError("expected runtimeconfig Error for value + text")